# 02 — Train Baseline: `tomato_ripeness_v1` (Kaggle, GPU)

**Mục tiêu:** train baseline YOLO nano trên `tomato_ripeness_v1`, lấy `best.pt` + đầy đủ evidence (train config, metrics, confusion matrix) để đối chiếu với mục tiêu MVP, đúng quy trình mục 9 trong `TONG_HOP_YEU_CAU_VA_DE_XUAT_AI_SMART_GREENHOUSE.docx`.

## Trước khi chạy
1. **Add Input** → chọn Kaggle Dataset output đã Save Version của `01_build_tomato_ripeness_v1.ipynb` (chứa `train/`, `val/`, `test/`, `test_outdomain_openfield/`, `data.yaml`).
2. Panel phải → **Accelerator: GPU** (P100 hoặc T4 x2 đều được — bắt buộc, notebook này train thật). **Internet: ON** (tải `ultralytics` + pretrained weights).
3. Session GPU trên Kaggle có giới hạn ~9–12 giờ/tuần tùy tài khoản — theo dõi để tránh hết quota giữa chừng.

## Cấu hình baseline (theo tài liệu, mục 9)
| Tham số | Giá trị |
|---|---|
| Model | YOLOv8n (`yolov8n.pt`, pretrained COCO) |
| imgsz | 640 |
| epochs | 100 (patience=20 sẽ tự dừng sớm nếu val không cải thiện) |
| batch | 16 (giảm xuống 8 nếu gặp CUDA OOM) |
| seed | 42 |
| pretrained | true |

## Việc notebook này làm
1. Tự nhận diện Kaggle Dataset input (`data.yaml` khớp 3 class ripeness), kiểm tra GPU khả dụng.
2. Train YOLOv8n baseline trên `train/val`.
3. Đánh giá **riêng biệt** trên `test` (cùng phân bố với train/val) và `test_outdomain_openfield` (nguồn `openfield_bd`, ngoài miền, license chưa xác nhận — chỉ dùng đánh giá, không train) để đo domain gap.
4. Lưu `train_config.yaml`, bảng so sánh metrics, và `best.pt` vào `/kaggle/working/` — sẵn sàng Save Version.
5. Vẽ dự đoán mẫu trên vài ảnh test để kiểm tra trực quan trước khi tin tưởng số liệu.

## Mục tiêu MVP tham khảo (không phải cam kết — báo cáo trung thực số liệu thật)
Precision ≥ 0.80, Recall ≥ 0.75, mAP@0.5 ≥ 0.80 trên `test`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=False)

import torch

print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "Không có GPU. Vào Settings (panel phải) -> Accelerator -> chọn GPU T4 x2 hoặc P100, "
        "rồi chạy lại từ đầu. Train YOLO trên CPU sẽ quá chậm để dùng được."
    )

### Bước 1 — Tự nhận diện Kaggle Dataset input
Tìm `data.yaml` khớp đúng 3 class của `tomato_ripeness_v1` trong `/kaggle/input` (không đoán đường dẫn — giống cách làm ở `01b_dataset_report_ripeness.ipynb`).

In [ ]:
from pathlib import Path

import yaml

TARGET_CLASSES = ["fruit_green_unripe", "fruit_turning", "fruit_ripe"]


def find_dataset_root():
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


DATASET_ROOT = find_dataset_root()
if DATASET_ROOT is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 3 class của tomato_ripeness_v1 trong /kaggle/input.\n"
        "Kiểm tra đã Add Input đúng Kaggle Dataset xuất ra từ 01_build_tomato_ripeness_v1.ipynb "
        "(SAU KHI notebook đó đã Save Version) chưa."
    )

DATA_YAML = DATASET_ROOT / "data.yaml"
OUTDOMAIN_DIR = DATASET_ROOT / "test_outdomain_openfield"

print("DATASET_ROOT:", DATASET_ROOT)
print("DATA_YAML   :", DATA_YAML)
print("test_outdomain_openfield tồn tại:", OUTDOMAIN_DIR.exists(),
      f"({len(list((OUTDOMAIN_DIR / 'images').glob('*')))} ảnh)" if OUTDOMAIN_DIR.exists() else "")

### Bước 2 — Vá lại `data.yaml` (quan trọng)
`data.yaml` gốc có trường `path:` là đường dẫn tuyệt đối `/kaggle/working/...` từ session của notebook đã tạo ra nó (`01_build_tomato_ripeness_v1.ipynb`) — session đó không còn tồn tại. Nếu dùng nguyên văn, `ultralytics` sẽ tìm ảnh sai chỗ và báo thiếu dữ liệu. Ghi lại một bản `data.yaml` mới trỏ đúng `DATASET_ROOT` hiện tại vào `/kaggle/working/`.

In [ ]:
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

orig_yaml = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("data.yaml gốc (path cũ có thể không còn tồn tại):")
print(yaml.safe_dump(orig_yaml, allow_unicode=True, sort_keys=False))

fixed_yaml = {
    "path": str(DATASET_ROOT),
    "train": orig_yaml.get("train", "train/images"),
    "val": orig_yaml.get("val", "val/images"),
    "test": orig_yaml.get("test", "test/images"),
    "names": orig_yaml["names"],
}
FIXED_DATA_YAML = WORKING / "data_ripeness.yaml"
with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_yaml, f, allow_unicode=True, sort_keys=False)

print("\ndata.yaml đã vá (dùng để train):")
print(FIXED_DATA_YAML.read_text(encoding="utf-8"))

# Sanity check: đếm ảnh thật theo từng path đã khai báo, tránh train "thành công" trên 0 ảnh.
for split_key in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / fixed_yaml[split_key]
    n = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    print(f"  {split_key:6s}: {n} ảnh tại {img_dir}")
    if n == 0:
        raise FileNotFoundError(f"{split_key} rỗng tại {img_dir} -> kiểm tra lại DATASET_ROOT/data.yaml.")

### Bước 3 — Train YOLOv8n baseline
`/kaggle/input` chỉ đọc — `ultralytics` bản mới tự bỏ qua lỗi ghi file `.cache` nhãn ở đó (log warning, không dừng train). Nếu bản `ultralytics` đang dùng vẫn lỗi cứng vì việc này, giải pháp là copy `DATASET_ROOT` sang `/kaggle/working/` rồi trỏ `FIXED_DATA_YAML` vào bản copy đó.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "tomato_ripeness_v1_yolov8n_baseline"
RUNS_DIR = WORKING / "runs"

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(FIXED_DATA_YAML),
    imgsz=640,
    epochs=100,
    batch=16,
    patience=20,
    seed=42,
    pretrained=True,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
print("\nbest.pt:", BEST_PT, "tồn tại:", BEST_PT.exists())

### Bước 4 — Đánh giá trên `test` (cùng phân bố với train/val)
Dùng `best.pt` (checkpoint tốt nhất theo val, không phải checkpoint cuối) để đánh giá khách quan trên `test` — tập này chưa từng được model thấy trong lúc train/chọn hyperparameter.

In [ ]:
best_model = YOLO(str(BEST_PT))

test_metrics = best_model.val(
    data=str(FIXED_DATA_YAML),
    split="test",
    imgsz=640,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_eval_test",
    exist_ok=True,
)

print("== Kết quả trên test (cùng phân bố) ==")
print(f"Precision (mean): {test_metrics.box.mp:.4f}")
print(f"Recall (mean)   : {test_metrics.box.mr:.4f}")
print(f"mAP@0.5         : {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95    : {test_metrics.box.map:.4f}")
print("\nTheo từng class:")
for i, cname in enumerate(TARGET_CLASSES):
    print(f"  {cname:20s} P={test_metrics.box.p[i]:.4f}  R={test_metrics.box.r[i]:.4f}  "
          f"AP50={test_metrics.box.ap50[i]:.4f}  AP50:95={test_metrics.box.ap[i]:.4f}")

### Bước 5 — Đánh giá trên `test_outdomain_openfield` (đo domain gap)
`test_outdomain_openfield` (nguồn `openfield_bd`) **không** nằm trong `data.yaml` gốc (license chưa xác nhận, chỉ dùng đánh giá). Tạo 1 yaml tạm trỏ `val` sang thư mục này để `model.val()` chạy được, không đụng tới `FIXED_DATA_YAML` đang dùng cho train/test chính.

In [ ]:
outdomain_metrics = None

if not OUTDOMAIN_DIR.exists() or not any((OUTDOMAIN_DIR / "images").glob("*")):
    print("Không có test_outdomain_openfield trong dataset này -> bỏ qua đánh giá domain gap.")
else:
    outdomain_yaml = {
        "path": str(DATASET_ROOT),
        "train": fixed_yaml["train"],  # bắt buộc phải có key nhưng không dùng tới
        "val": "test_outdomain_openfield/images",
        "names": fixed_yaml["names"],
    }
    OUTDOMAIN_DATA_YAML = WORKING / "data_ripeness_outdomain.yaml"
    with open(OUTDOMAIN_DATA_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(outdomain_yaml, f, allow_unicode=True, sort_keys=False)

    outdomain_metrics = best_model.val(
        data=str(OUTDOMAIN_DATA_YAML),
        split="val",
        imgsz=640,
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_eval_outdomain",
        exist_ok=True,
    )

    print("== Kết quả trên test_outdomain_openfield (ngoài miền) ==")
    print(f"Precision (mean): {outdomain_metrics.box.mp:.4f}")
    print(f"Recall (mean)   : {outdomain_metrics.box.mr:.4f}")
    print(f"mAP@0.5         : {outdomain_metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95    : {outdomain_metrics.box.map:.4f}")
    print("\nTheo từng class:")
    for i, cname in enumerate(TARGET_CLASSES):
        print(f"  {cname:20s} P={outdomain_metrics.box.p[i]:.4f}  R={outdomain_metrics.box.r[i]:.4f}  "
              f"AP50={outdomain_metrics.box.ap50[i]:.4f}  AP50:95={outdomain_metrics.box.ap[i]:.4f}")

### Bước 6 — So sánh test vs. test_outdomain_openfield

In [ ]:
import pandas as pd

rows = [{
    "eval_set": "test (cùng miền)",
    "precision": round(float(test_metrics.box.mp), 4),
    "recall": round(float(test_metrics.box.mr), 4),
    "map50": round(float(test_metrics.box.map50), 4),
    "map50_95": round(float(test_metrics.box.map), 4),
}]
if outdomain_metrics is not None:
    rows.append({
        "eval_set": "test_outdomain_openfield (ngoài miền)",
        "precision": round(float(outdomain_metrics.box.mp), 4),
        "recall": round(float(outdomain_metrics.box.mr), 4),
        "map50": round(float(outdomain_metrics.box.map50), 4),
        "map50_95": round(float(outdomain_metrics.box.map), 4),
    })

compare_df = pd.DataFrame(rows)
print(compare_df.to_string(index=False))

if outdomain_metrics is not None:
    gap_map50 = rows[0]["map50"] - rows[1]["map50"]
    print(f"\nChênh lệch mAP@0.5 (test - outdomain): {gap_map50:.4f}")
    if gap_map50 > 0.15:
        print("[CẢNH BÁO] Chênh lệch domain gap khá lớn (>0.15) — model có thể không tổng quát tốt "
              "ra môi trường ngoài đồng/camera khác. Cân nhắc bổ sung ảnh thật từ camera IMX179 khi có, "
              "theo đúng khuyến nghị của tài liệu (mục 'Ưu tiên bổ sung ảnh từ chính camera').")
    else:
        print("Chênh lệch domain gap ở mức chấp nhận được cho baseline.")

MVP_TARGETS = {"precision": 0.80, "recall": 0.75, "map50": 0.80}
print("\nSo với mục tiêu MVP tham khảo trên test (không phải cam kết):")
for k, target in MVP_TARGETS.items():
    actual = rows[0][k]
    status = "ĐẠT" if actual >= target else "CHƯA ĐẠT"
    print(f"  {k}: {actual:.4f} (mục tiêu {target}) -> {status}")

### Bước 7 — Lưu train config, metrics và `best.pt` gọn để Save Version
Theo mục 10 của tài liệu: `best.pt` một mình không đủ, cần giữ kèm dataset version, class mapping, train config và metrics.

In [ ]:
import shutil

MODELS_DIR = WORKING / "models"
REPORTS_DIR = WORKING / "reports" / RUN_NAME
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, MODELS_DIR / "tomato_ripeness_yolov8n_baseline.pt")

train_config = {
    "model": "yolov8n.pt",
    "data_yaml": str(FIXED_DATA_YAML),
    "dataset_root": str(DATASET_ROOT),
    "classes": TARGET_CLASSES,
    "imgsz": 640,
    "epochs": 100,
    "batch": 16,
    "patience": 20,
    "seed": 42,
    "pretrained": True,
    "run_name": RUN_NAME,
}
with open(REPORTS_DIR / "train_config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_config, f, allow_unicode=True, sort_keys=False)

compare_df.to_csv(REPORTS_DIR / "metrics_test_vs_outdomain.csv", index=False)

# Bản đầy đủ theo từng class (test) để tiện đối chiếu sau này.
per_class_rows = [{
    "class": cname,
    "precision": round(float(test_metrics.box.p[i]), 4),
    "recall": round(float(test_metrics.box.r[i]), 4),
    "ap50": round(float(test_metrics.box.ap50[i]), 4),
    "ap50_95": round(float(test_metrics.box.ap[i]), 4),
} for i, cname in enumerate(TARGET_CLASSES)]
pd.DataFrame(per_class_rows).to_csv(REPORTS_DIR / "metrics_per_class_test.csv", index=False)

print("Đã lưu:")
print(" -", MODELS_DIR / "tomato_ripeness_yolov8n_baseline.pt")
print(" -", REPORTS_DIR / "train_config.yaml")
print(" -", REPORTS_DIR / "metrics_test_vs_outdomain.csv")
print(" -", REPORTS_DIR / "metrics_per_class_test.csv")
print(f" - Toàn bộ log/plot train (loss curve, PR curve, confusion matrix): {RUNS_DIR / RUN_NAME}")

### Bước 8 — Xác nhận trực quan: dự đoán thật trên ảnh mẫu
Số liệu mAP/Precision/Recall có thể "đẹp" nhưng vẫn cần nhìn trực tiếp dự đoán trên ảnh — đúng nguyên tắc đã áp dụng xuyên suốt dự án này (không tin số liệu mà không xác nhận trực quan).

In [ ]:
import random

import matplotlib.pyplot as plt

random.seed(42)


def show_predictions(img_dir, n=6, title=""):
    img_dir = Path(img_dir)
    all_imgs = sorted(img_dir.glob("*"))
    if not all_imgs:
        print(f"[{title}] Không có ảnh trong {img_dir}.")
        return
    sample = random.sample(all_imgs, min(n, len(all_imgs)))
    results = best_model.predict(source=[str(p) for p in sample], imgsz=640, conf=0.25, verbose=False)

    fig, axes = plt.subplots(1, len(sample), figsize=(3.6 * len(sample), 3.6))
    axes = [axes] if len(sample) == 1 else axes
    for ax, res, img_path in zip(axes, results, sample):
        annotated = res.plot()[:, :, ::-1]  # BGR -> RGB
        ax.imshow(annotated)
        ax.set_title(img_path.name, fontsize=6)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    save_path = REPORTS_DIR / f"predictions_{title.replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=80, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Đã lưu:", save_path)


show_predictions(DATASET_ROOT / "test" / "images", title="test_cung_mien")
if OUTDOMAIN_DIR.exists():
    show_predictions(OUTDOMAIN_DIR / "images", title="test_outdomain_openfield")

## Kết quả
- `models/tomato_ripeness_yolov8n_baseline.pt` — checkpoint tốt nhất theo val.
- `reports/tomato_ripeness_v1_yolov8n_baseline/` — `train_config.yaml`, `metrics_test_vs_outdomain.csv`, `metrics_per_class_test.csv`, ảnh dự đoán mẫu.
- `runs/tomato_ripeness_v1_yolov8n_baseline/` — toàn bộ log train của ultralytics: loss curve (`results.png`), `confusion_matrix.png`, `PR_curve.png`, `args.yaml`.

## Trước khi Save Version — checklist
- [ ] Bước 6: Precision/Recall/mAP@0.5 trên `test` — đối chiếu với mục tiêu MVP tham khảo, ghi nhận trung thực dù đạt hay chưa.
- [ ] Bước 6: chênh lệch domain gap giữa `test` và `test_outdomain_openfield` — nếu lớn, đây là tín hiệu cần thêm ảnh thật từ camera IMX179 trước khi triển khai.
- [ ] Bước 8: ảnh dự đoán mẫu — box có khoanh đúng quả, nhãn màu đúng độ chín không (không chỉ tin số liệu).
- [ ] Xem `runs/.../confusion_matrix.png` để biết model hay nhầm lẫn giữa 2 class nào nhất (thường là `fruit_turning` với 2 class còn lại, do ranh giới màu sắc mờ).

## Bước tiếp theo
- Nếu số liệu đạt mục tiêu: cân nhắc train phiên bản chính thức (tăng dữ liệu/epoch có kiểm soát, không chỉ tăng epoch mù quáng — theo đúng khuyến nghị tài liệu).
- Nếu domain gap lớn: ưu tiên thu thập ảnh thật từ camera IMX179 trong nhà kính trước khi train lại.
- Nhánh còn lại: `03_train_leaf_baseline.ipynb` cho `tomato_leaf_disease_v1` (cấu hình tương tự, đánh giá trên `test`, không có tập ngoài miền riêng).